In [0]:

import pandas as pd
from pyspark.sql import SparkSession

# reading data from source(Github - Sales) (copy link from file - Raw)
df = pd.read_csv("https://raw.githubusercontent.com/Bhevendra/ML-Datasets/refs/heads/main/retail_data/sales.csv")

# creating sparksession
spark = SparkSession.builder.getOrCreate()


# changing spark dataframe to spark dataframe
df = spark.createDataFrame(df)



## EXPLAINATION OF CODE BELOW : 

* importing col = used to reference a column 
* from_json  = convert json string -- structured columns
* schema_of_json = automatically detects json schema
#### why = 
bcz spark cannot understand a json string directly -- it need schema to parse (analyze/ understand) it 

### Get a sample_json
* df.select("product") = take only product column
* .first() = take only first row
* [0] = extract value from this row
* RESULT = one json string sample
* WHy = spark need one json sampl e to understand the structure

### INFER SCHEMa
* json_schema = schema_of_json(sample_json) ====== it automatically generates schema like 
struct<name: string, price:int>
* Writing schema manually is hard
* json may have many fields and this make code dynamic + scalable

### Convert json - Struct (parse it)
* df_flat = df.withColumn("product_json", from_json(col("product"), json_schema)) \
    

* it takes json string column (product) and convert it to structured column (struct type)
* why = so spark can now access fields inside json
* Result = before --------- product = {"name": "Laptop", "price": 50000}
* Result = after --------- product_json = {name: Laptop, price: 50000}


### Flatten JSON file (bcz we want flat files for anayltics not nested file)

* .select("*", "product_json.*") \
* (*) keep all original columns
* product_json.* = expland struct into columns ( now = name | price) 


1. What is Nested Data

👉 Data inside data (hierarchical structure)

Example:
{
  "customer_id": 1,
  "product": {
    "name": "Laptop",
    "price": 50000
  }
}
💡 Meaning:
product is not a simple column
It contains another structure (object)

👉 This is called nested (struct)

🔹 2. What is Flat Data

👉 Everything is in simple columns (no hierarchy)

Example:
customer_id | name   | price
------------|--------|-------
1           | Laptop | 50000
💡 Meaning:
No nested objects
Each field is a separate column

👉 This is called flat (tabular format)

🔹 3. Why Nested Data Comes

👉 Because of real-world data sources:

APIs → JSON response
Kafka → streaming JSON
Logs → semi-structured data
MongoDB → document format
💡 Example:

APIs naturally return:

{
  "order": {
    "id": 101,
    "items": [...]
  }
}

👉 That’s nested by design

🔹 4. Why We Convert Nested → Flat (VERY IMPORTANT)

We flatten because:

❌ Problems with Nested:
Hard to query in SQL
Difficult joins
Poor performance in analytics
Not BI-friendly (Power BI, Tableau)
✅ Benefits of Flat:
Easy SQL queries
Faster aggregations
Better for reporting
Clean schema (Gold layer)
🔹 5. How Flattening Works (Your Code Context)

👉 Your code is doing exactly this:

Step-by-step:
Take JSON string
Convert → struct (nested object)
Expand fields using:
.select("*", "product_json.*")
Remove original column



#### drop unwanted columns 

* .drop("product", "product_json")
* removes original json column and intermediate struct column 


### We extract JSON schema dynamically and flatten it into columns so spark can process it efficiently


























In [0]:
from pyspark.sql.functions import col, from_json, schema_of_json

# sample one JSON string
sample_json = df.select("product").filter(col("product").isNotNull()).first()[0]

# infer schema
json_schema = schema_of_json(sample_json)

# flatten
df_flat = df.withColumn("product_json", from_json(col("product"), json_schema)) \
    .select("*", "product_json.*") \
    .drop("product", "product_json")

display(df_flat)

In [0]:
df_flat.write.format("delta").mode("overwrite").saveAsTable("batch_1.data.sales")